# 1D Additive-Noise Goodness-of-Fit Toy

This notebook is a thin driver over `multivariate_crossfit_core` for the one-dimensional additive-noise GOF experiment.

It keeps the statistical choices explicit:

- data are generated as `Y = E[Y | X] + eps`, with `rho = 0` homoskedastic noise and `rho = 0.35` heteroskedastic noise,
- both directions, `Y on X` and `X on Y`, are tested,
- the first-stage mean is Gaussian-kernel KRR with 5-fold CV inside each training complement,
- the HSIC test kernels are Gaussian with median-heuristic bandwidths,
- cross-fit permutation p-values use U-statistics and permute/recenter the X kernel,
- debiased and non-debiased cross-fit estimators also get V-statistic bootstrap confidence sets for `||psi||`.


See `CAUSAL_COVARIATE_REPRODUCTION.md`. For saved-data analysis only, run the imports cell, then the two cells under **Load And Visualize Saved Runs**. Select a specific saved payload in that section. The paper uses 100 core repetitions, n=250 and 399 draws.

In [ ]:
from __future__ import annotations

from functools import partial
import time

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from multivariate_crossfit_core import (
    RunConfig,
    bootstrap_sqrt_hsic_ci_crossfit,
    build_fold_masks,
    center_cross_block,
    compute_xi_debias_term,
    crossfit_u_from_blocks,
    fit_mean_forecasts,
    fit_v_weights,
    fold_indices_from_masks,
    gaussian_gram,
    hsic_unbiased,
    median_heuristic,
    permuted_naive_hsic_u_by_x,
    standard_residual_hsic_tests,
    validate_config,
)

plt.style.use("seaborn-v0_8-whitegrid")

from pathlib import Path
from IPython.display import display

# Register the paper font explicitly: an old matplotlib cache can omit system fonts.
from matplotlib import font_manager
for font_path in font_manager.findSystemFonts():
    if Path(font_path).name.startswith("LiberationSans"):
        font_manager.fontManager.addfont(font_path)
font_manager.findfont("Liberation Sans", fallback_to_default=False)
plt.rcParams["font.family"] = "Liberation Sans"

CAUSAL_FIGURE_DIR = Path("figures/causal_arrow")


## Data

The forward structural model is homoskedastic when `rho = 0` and heteroskedastic when `rho = 0.35`. The reverse direction is formed by swapping predictor and response; there is no structural reverse signal used by the test.


In [ ]:
BASE_STD = 0.45
RHO_VALUES = (0.0, 0.35)
DIRECTIONS = ("Y on X", "X on Y")
SEED = 12_345

_x_grid_for_scale = np.linspace(-2.0, 2.0, 20_001)


def f_true(x):
    x = np.asarray(x, dtype=float)
    # Deliberately non-invertible-ish on [-2, 2]: harder for KRR to learn and
    # much less likely to admit an additive-noise explanation in the reverse direction.
    return 0.55 * x + 0.95 * np.sin(1.65 * x) + 0.38 * np.sin(3.6 * x) + 0.10 * x**3


def eps_shape_raw(x):
    x = np.asarray(x, dtype=float)
    return np.sin(1.6 * x) + 0.35 * np.cos(2.4 * x)


_EPS_SHAPE_MEAN = eps_shape_raw(_x_grid_for_scale).mean()
_EPS_SHAPE_STD = eps_shape_raw(_x_grid_for_scale).std()


def eps_shape(x):
    return (eps_shape_raw(x) - _EPS_SHAPE_MEAN) / _EPS_SHAPE_STD


def sigma_fn(x, rho):
    return BASE_STD * np.exp(rho * eps_shape(x))


def simulate_anm(n, rho, seed):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-2.0, 2.0, size=n)
    sigma = sigma_fn(x, rho)
    eps = sigma * rng.normal(size=n)
    y = f_true(x) + eps
    return {"x": x, "y": y, "eps": eps, "sigma": sigma, "rho": rho, "signal_y_on_x": f_true(x)}


def direction_arrays(data, direction):
    if direction == "Y on X":
        return data["x"], data["y"], data["signal_y_on_x"]
    if direction == "X on Y":
        return data["y"], data["x"], np.zeros_like(data["x"])
    raise ValueError(direction)


def direction_latex(direction):
    if direction == "Y on X":
        return r"$Y$ on $X$"
    if direction == "X on Y":
        return r"$X$ on $Y$"
    raise ValueError(direction)


def null_status_label(rho, direction):
    return "null holds" if direction == "Y on X" and np.isclose(rho, 0.0) else "null fails"


def panel_title(rho, direction):
    return f"{direction_latex(direction)}: {null_status_label(rho, direction)}"


def compact_layout(fig):
    fig.set_constrained_layout_pads(w_pad=0.02, h_pad=0.02, wspace=0.02, hspace=0.04)


## Configuration

`N_FOLDS = 5` is the faithful setting. Changing it to `2` is fine for quick smoke tests, provided `N` is still divisible by `N_FOLDS` and each fold complement is divisible by the nuisance CV fold count.


In [ ]:
N_FOLDS = 5
SPLIT_PREFIXES = tuple(range(1, N_FOLDS))
SPLIT_LABEL = {prefix: f"split {100 * prefix // N_FOLDS}%" for prefix in SPLIT_PREFIXES}


def make_core_config(n, bootstrap_B):
    config = RunConfig(
        N=n,
        num_folds=N_FOLDS,
        dimx=1,
        dimy=1,
        mean_kernel="gaussian",
        mean_cv_folds=5,
        v_cv_folds=5,
        mean_alphas=(1e-3, 1e-2, 1e-1, 1.0, 10.0),
        mean_length_scales=(0.5, 1.0, 2.0, 4.0, 8.0),
        v_bandwidth_multipliers=(0.5, 1.0, 2.0),
        v_etas=(1e-4, 1e-3, 1e-2),
        v_nus=(0.5, 1.5, 2.5),
        v_fit_mode="residualized_centered",
        bootstrap_B=bootstrap_B,
        ci_alpha=0.05,
        bootstrap_seed=SEED,
        compute_debiased_objects=True,
        compute_v_diagnostics=False,
        compute_crossfit_rate_diagnostics=False,
        compute_plugin_bias_diagnostic=False,
        compute_bootstrap_intervals=False,
        compute_delta_intervals=False,
        compute_population_oracle=False,
        compute_permutation_test=False,
        compute_standard_residual_hsic_tests=True,
        standard_residual_hsic_fit_prefixes=SPLIT_PREFIXES,
    )
    validate_config(config)
    return config


DEMO_N = 250
DEMO_B = 399
DEMO_CONFIG = make_core_config(DEMO_N, DEMO_B)

# Exact repetitions rerun sklearn CV, v-CV, permutation p-values, and
# reverse-triangle bootstrap CIs. This is the faithful repeated procedure.
RUN_CORE_REPETITIONS = True
CORE_N_REPS = 100
CORE_N = 250
CORE_B = 399
CORE_CONFIG = make_core_config(CORE_N, CORE_B)

# Fast repetitions are only a cheap naive diagnostic with fixed Gaussian KRR
# bandwidths. They are off by default because they do not include debiasing or
# reverse-triangle bootstrap inference.
RUN_FAST_REPETITIONS = False
FAST_N_REPS = 500
FAST_N = 200
FAST_B = 199
FAST_KRR_ETA = 1e-2
FAST_MEAN_ALPHA = {
    "Y on X": 0.90,
    "X on Y": 1.80,
}


## Cross-Fit Test

The residual-side blocks are built from fold-specific residuals. For the permutation null, the full X Gram matrix is permuted within folds and then recentered for that permutation before forming the U-statistic.


In [ ]:
@partial(jax.jit, static_argnames=("size_fold", "n_perm"))
def permutation_pvalue_crossfit_by_x(H_folds, K, WXi_side_folds, fold_indices, size_fold, n_perm, seed):
    observed = crossfit_u_from_blocks(H_folds)
    keys = jax.random.split(jax.random.PRNGKey(seed), n_perm)
    null = jax.lax.map(
        lambda subkey: permuted_naive_hsic_u_by_x(K, WXi_side_folds, fold_indices, size_fold, subkey),
        keys,
    )
    p_value = (1.0 + jnp.sum(null >= observed)) / (n_perm + 1.0)
    return {
        "statistic_u": observed,
        "sqrt_hsic_u": jnp.sqrt(jnp.maximum(observed, 0.0)),
        "p_value": p_value,
        "null_mean": jnp.mean(null),
    }


def _col(a):
    return jnp.asarray(np.asarray(a, dtype=float)[:, None])


def _float(x):
    return float(np.asarray(x))


def _bootstrap_summary(H_folds, config, size_fold, seed):
    interval = bootstrap_sqrt_hsic_ci_crossfit(
        H_folds,
        config.num_folds,
        size_fold,
        config.bootstrap_B,
        config.ci_alpha,
        seed,
    )
    return {
        "sqrt_hsic_v": _float(interval["sqrt_hsic"]),
        "bootstrap_p_value": _float(interval["p_value"]),
        "zero_inclusion_quantile": _float(interval["zero_inclusion_quantile"]),
        "ci_lower": _float(interval["ci_lower"]),
        "ci_upper": _float(interval["ci_upper"]),
        "ci_radius": _float(interval["radius"]),
    }


def _permutation_summary(out):
    return {
        "p_value": _float(out["p_value"]),
        "sqrt_hsic_u": _float(out["sqrt_hsic_u"]),
        "null_mean": _float(out["null_mean"]),
    }


def full_crossfit_core_test(predictor, response, signal_for_diag, config, seed):
    X = _col(predictor)
    Y = _col(response)
    signal = _col(signal_for_diag)
    fold_masks = build_fold_masks(config.N, config.num_folds)
    size_fold = config.N // config.num_folds
    fold_indices = fold_indices_from_masks(fold_masks, size_fold)

    mean_forecasts, mean_diagnostics = fit_mean_forecasts(config, X, Y, signal, fold_masks)
    Xi_hat_folds = Y[None, :, :] - mean_forecasts
    Xi_hat_oos = jnp.concatenate([Xi_hat_folds[k][mask] for k, mask in enumerate(fold_masks)], axis=0)

    alpha_x = jnp.maximum(median_heuristic(X, X), 1e-12)
    K = gaussian_gram(X, X, alpha_x)

    alpha_xi = jnp.maximum(median_heuristic(Xi_hat_oos, Xi_hat_oos), 1e-12)
    WX_folds = []
    WXi_folds = []
    for k, mask_k in enumerate(fold_masks):
        wx_row = []
        wxi_row = []
        Xi_k = Xi_hat_folds[k]
        for l, mask_l in enumerate(fold_masks):
            Xi_l = Xi_hat_folds[l]
            wx_row.append(center_cross_block(K, mask_k, mask_l))
            L_kl = gaussian_gram(Xi_k, Xi_l, alpha_xi)
            wxi_row.append(center_cross_block(L_kl, mask_k, mask_l))
        WX_folds.append(wx_row)
        WXi_folds.append(wxi_row)
    WX_folds = jnp.asarray(WX_folds)
    WXi_folds = jnp.asarray(WXi_folds)

    _, W_oos_folds, v_params = fit_v_weights(config, X, Xi_hat_folds, fold_masks, alpha_xi)
    xi_debias_term = compute_xi_debias_term(config, Xi_hat_folds, fold_masks, W_oos_folds, alpha_xi)

    naive_D = WX_folds * WXi_folds
    WXi_debiased_folds = WXi_folds + xi_debias_term
    debiased_D = WX_folds * WXi_debiased_folds

    naive_perm = permutation_pvalue_crossfit_by_x(
        naive_D, K, WXi_folds, fold_indices, size_fold, config.bootstrap_B, seed
    )
    debiased_perm = permutation_pvalue_crossfit_by_x(
        debiased_D, K, WXi_debiased_folds, fold_indices, size_fold, config.bootstrap_B, seed + 1_000
    )

    fold_fitted_values = np.asarray(mean_forecasts[:, :, 0])
    fitted_oos = np.empty(config.N)
    residual_oos = np.empty(config.N)
    for k, mask in enumerate(np.asarray(fold_masks)):
        fitted_oos[mask] = np.asarray(mean_forecasts[k][mask, 0])
        residual_oos[mask] = np.asarray(Xi_hat_folds[k][mask, 0])

    return {
        "naive": {
            **_permutation_summary(naive_perm),
            **_bootstrap_summary(naive_D, config, size_fold, seed + 2_000),
        },
        "debiased": {
            **_permutation_summary(debiased_perm),
            **_bootstrap_summary(debiased_D, config, size_fold, seed + 3_000),
        },
        "fold_fitted_values": fold_fitted_values,
        "fitted_oos": fitted_oos,
        "residual_oos": residual_oos,
        "alpha_x": _float(alpha_x),
        "alpha_xi": _float(alpha_xi),
        "mean_diagnostics": mean_diagnostics,
        "v_params": v_params,
    }


def split_core_tests(predictor, response, signal_for_diag, config):
    X = _col(predictor)
    Y = _col(response)
    signal = _col(signal_for_diag)
    fold_masks = build_fold_masks(config.N, config.num_folds)
    rows = standard_residual_hsic_tests(config, X, Y, signal, fold_masks)
    out = []
    for row in rows:
        prefix = int(row["fit_prefix_folds"])
        out.append({
            "method": f"{SPLIT_LABEL[prefix]} naive",
            "fit_fraction": prefix / config.num_folds,
            "n_fit": int(row["n_fit"]),
            "n_test": int(row["n_test"]),
            "p_value": _float(row["naive_hsic_u_p_value"]),
            "sqrt_hsic_u": _float(row["naive_sqrt_hsic_u"]),
        })
        out.append({
            "method": f"{SPLIT_LABEL[prefix]} debiased",
            "fit_fraction": prefix / config.num_folds,
            "n_fit": int(row["n_fit"]),
            "n_test": int(row["n_test"]),
            "p_value": _float(row["debiased_hsic_u_p_value"]),
            "sqrt_hsic_u": _float(row["debiased_sqrt_hsic_u"]),
        })
    return out


## One Draw

The table shows U-statistic permutation p-values. The full cross-fit rows also show the V-statistic bootstrap confidence set for `||psi||`.


In [ ]:
def run_one_draw(n, config, seed_base):
    datasets = {}
    diagnostics = {}
    rows = []
    for rho_idx, rho in enumerate(RHO_VALUES):
        data = simulate_anm(n, rho, seed=seed_base + 100 * rho_idx)
        datasets[rho] = data
        for dir_idx, direction in enumerate(DIRECTIONS):
            predictor, response, signal = direction_arrays(data, direction)
            full = full_crossfit_core_test(
                predictor, response, signal, config, seed=seed_base + 10_000 + 100 * rho_idx + dir_idx
            )
            diagnostics[(rho, direction)] = full
            rows.append({
                "rho": rho,
                "direction": direction,
                "method": "full crossfit naive",
                "n_fit": n,
                "n_test": n,
                **full["naive"],
            })
            rows.append({
                "rho": rho,
                "direction": direction,
                "method": "full crossfit debiased",
                "n_fit": n,
                "n_test": n,
                **full["debiased"],
            })
            for split_row in split_core_tests(predictor, response, signal, config):
                rows.append({"rho": rho, "direction": direction, **split_row})
    return datasets, diagnostics, rows


def print_results_table(rows):
    header = (
        f"{'rho':>4}  {'direction':<7}  {'method':<25}  {'n_fit':>5}  {'n_test':>6}  "
        f"{'sqrtU':>8}  {'perm p':>7}  {'boot p':>7}  {'sqrtV CI':>21}"
    )
    print(header)
    print("-" * len(header))
    for row in rows:
        if "ci_lower" in row:
            ci = f"[{row['ci_lower']:.4f}, {row['ci_upper']:.4f}]"
            boot_p = f"{row['bootstrap_p_value']:.4f}"
        else:
            ci = ""
            boot_p = ""
        print(
            f"{row['rho']:>4.2f}  {row['direction']:<7}  {row['method']:<25}  "
            f"{row['n_fit']:>5d}  {row['n_test']:>6d}  {row['sqrt_hsic_u']:>8.4f}  "
            f"{row['p_value']:>7.4f}  {boot_p:>7}  {ci:>21}"
        )


def print_mean_cv_summary(diagnostics):
    print("\nMean KRR CV choices by outer fold")
    header = f"{'rho':>4}  {'direction':<7}  {'fold':>4}  {'kernel':<8}  {'alpha':>9}  {'length':>9}  {'cv_mse':>10}"
    print(header)
    print("-" * len(header))
    for (rho, direction), info in diagnostics.items():
        for diag in info["mean_diagnostics"]:
            print(
                f"{rho:>4.2f}  {direction:<7}  {diag['fold']:>4d}  {diag['kernel']:<8}  "
                f"{diag['alpha']:>9.3g}  {diag['length_scale']:>9.3g}  {diag['cv_mse']:>10.4f}"
            )


def print_fold_fit_spread(diagnostics):
    print("\nFold-complement mean-fit disagreement on the observed predictor values")
    header = f"{'rho':>4}  {'direction':<7}  {'mean_sd':>9}  {'median_sd':>9}  {'max_sd':>9}"
    print(header)
    print("-" * len(header))
    for (rho, direction), info in diagnostics.items():
        fold_values = np.asarray(info["fold_fitted_values"])
        fold_sd = np.std(fold_values, axis=0)
        print(
            f"{rho:>4.2f}  {direction:<7}  {np.mean(fold_sd):>9.4f}  "
            f"{np.median(fold_sd):>9.4f}  {np.max(fold_sd):>9.4f}"
        )


datasets, diagnostics, one_draw_rows = run_one_draw(DEMO_N, DEMO_CONFIG, SEED)
print_results_table(one_draw_rows)
print_mean_cv_summary(diagnostics)
print_fold_fit_spread(diagnostics)


## Forward And Reverse Fits

The left column is the structural direction `Y on X`. The right column is the reverse regression problem `X on Y` on the same data.


In [ ]:
def plot_fold_mean_fits(ax, predictor, response, fit_info, title, xlabel, ylabel, true_grid=None, ylabel_side="left"):
    order = np.argsort(predictor)
    ax.scatter(predictor, response, s=16, alpha=0.28, color="#334155", label="sample")
    for fold_idx, fitted in enumerate(fit_info["fold_fitted_values"]):
        label = "fold fits" if fold_idx == 0 else None
        ax.plot(predictor[order], fitted[order], color="#7c3aed", linewidth=1.05, alpha=0.48, label=label)
    if true_grid is not None:
        grid, mean, upper, lower = true_grid
        ax.plot(grid, mean, color="#111827", linewidth=1.55, label=r"$m(x)$")
        ax.plot(grid, upper, color="#dc2626", linewidth=1.15, linestyle="--", label=r"$m(x)\pm 2\sigma(x)$")
        ax.plot(grid, lower, color="#dc2626", linewidth=1.15, linestyle="--")
    ax.set_title(title, fontsize=11, loc="left")
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, rotation=0, labelpad=8, va="center", fontsize=12)
    ax.tick_params(axis="both", labelsize=10)
    if ylabel_side == "right":
        ax.yaxis.set_label_position("right")
        ax.yaxis.set_label_coords(1.04, 0.5)
    else:
        ax.yaxis.set_label_coords(-0.05, 0.5)


fig, axes = plt.subplots(len(RHO_VALUES), 2, figsize=(8.9, 4.9), constrained_layout=False)
fig.subplots_adjust(left=0.075, right=0.94, bottom=0.16, top=0.93, wspace=0.1, hspace=0.4)
x_grid = np.linspace(-2.0, 2.0, 500)

for row_idx, rho in enumerate(RHO_VALUES):
    data = datasets[rho]
    true_grid = (
        x_grid,
        f_true(x_grid),
        f_true(x_grid) + 2 * sigma_fn(x_grid, rho),
        f_true(x_grid) - 2 * sigma_fn(x_grid, rho),
    )
    plot_fold_mean_fits(
        axes[row_idx, 0],
        data["x"],
        data["y"],
        diagnostics[(rho, "Y on X")],
        panel_title(rho, "Y on X"),
        r"$X$",
        r"$Y$",
        true_grid=true_grid,
        ylabel_side="left",
    )
    plot_fold_mean_fits(
        axes[row_idx, 1],
        data["y"],
        data["x"],
        diagnostics[(rho, "X on Y")],
        panel_title(rho, "X on Y"),
        r"$Y$",
        r"$X$",
        ylabel_side="right",
    )

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, fontsize=11, loc="lower center", ncol=4, bbox_to_anchor=(0.5, 0.005))

CAUSAL_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(CAUSAL_FIGURE_DIR / "causal_arrow_data.png", dpi=100, bbox_inches="tight")


## Residual Distributions

Forward panels show the true structural noise. Reverse panels show the cross-fitted residual from the reverse regression.


In [ ]:
def tertile_masks(z):
    q = np.quantile(z, [1 / 3, 2 / 3])
    return [z <= q[0], (z > q[0]) & (z <= q[1]), z > q[1]]


fig, axes = plt.subplots(len(RHO_VALUES), 2, figsize=(9.4, 4.9), constrained_layout=True)
compact_layout(fig)
colors = ["#2563eb", "#059669", "#dc2626"]
labels = ["low predictor", "middle predictor", "high predictor"]

for row_idx, rho in enumerate(RHO_VALUES):
    data = datasets[rho]
    panels = [
        ("Y on X", data["x"], data["eps"]),
        ("X on Y", data["y"], diagnostics[(rho, "X on Y")]["residual_oos"]),
    ]
    for col_idx, (direction, predictor, resid) in enumerate(panels):
        ax = axes[row_idx, col_idx]
        for mask, color, label in zip(tertile_masks(predictor), colors, labels):
            ax.hist(resid[mask], bins=28, density=True, histtype="step", linewidth=2.0, color=color, label=label)
        ax.set_title(panel_title(rho, direction), fontsize=11)
        ax.set_xlabel(r"residual / noise")
        ax.set_ylabel("density")
        if row_idx == 0 and col_idx == 1:
            ax.legend(frameon=False, fontsize=11)


## Repeated Calibration Diagnostics

There are two computation paths.

- `RUN_CORE_REPETITIONS` reruns the faithful cross-validated core computation, including full cross-fit naive/debiased permutation p-values and reverse-triangle bootstrap p-values/CIs. This is now the default repeated path.
- `RUN_FAST_REPETITIONS` is a fast naive diagnostic with fixed Gaussian-KRR mean bandwidths. It is useful for rough Monte Carlo shape checks, but it is not the full debiased/bootstrap procedure.

The bootstrap p-value is obtained by inverting the reverse-triangle confidence sets. If `s = ||hat psi||` and `D_b = ||hat psi - hat psi^(b)||^2`, then zero enters the CI once the bootstrap quantile radius exceeds `s`; the p-value is the empirical upper-tail probability `Pr_b(D_b >= s^2)`.


In [ ]:
@jax.jit
def fixed_gaussian_krr_predict(x_train, y_train, x_eval, alpha, eta):
    X_train = x_train[:, None]
    X_eval = x_eval[:, None]
    alpha = jnp.maximum(alpha, 1e-12)
    K_train = gaussian_gram(X_train, X_train, alpha)
    K_eval = gaussian_gram(X_eval, X_train, alpha)
    n_train = x_train.shape[0]
    coef = jnp.linalg.solve(K_train + eta * n_train * jnp.eye(n_train, dtype=K_train.dtype), y_train)
    return K_eval @ coef


@partial(jax.jit, static_argnames=("n_perm",))
def fast_permutation_hsic_pvalue(predictor, residual, n_perm, seed):
    X = predictor[:, None]
    R = residual[:, None]
    alpha_x = jnp.maximum(median_heuristic(X, X), 1e-12)
    alpha_r = jnp.maximum(median_heuristic(R, R), 1e-12)
    K = gaussian_gram(X, X, alpha_x)
    L = gaussian_gram(R, R, alpha_r)
    observed = hsic_unbiased(K, L)

    keys = jax.random.split(jax.random.PRNGKey(seed), n_perm)

    def one_perm(key):
        perm = jax.random.permutation(key, predictor.shape[0])
        return hsic_unbiased(K[perm][:, perm], L)

    null = jax.lax.map(one_perm, keys)
    return {
        "p_value": (1.0 + jnp.sum(null >= observed)) / (n_perm + 1.0),
        "sqrt_hsic_u": jnp.sqrt(jnp.maximum(observed, 0.0)),
    }


def fast_full_residuals(predictor, response, alpha, eta=FAST_KRR_ETA, num_folds=N_FOLDS):
    predictor = jnp.asarray(predictor, dtype=jnp.float64)
    response = jnp.asarray(response, dtype=jnp.float64)
    alpha = jnp.asarray(alpha, dtype=jnp.float64)
    n = len(response)
    folds = np.array_split(np.arange(n), num_folds)
    residual = np.empty(n)
    fitted = np.empty(n)
    all_idx = np.arange(n)
    for hold in folds:
        train = np.setdiff1d(all_idx, hold, assume_unique=False)
        pred = fixed_gaussian_krr_predict(predictor[train], response[train], predictor[hold], alpha, eta)
        pred_np = np.asarray(pred)
        fitted[hold] = pred_np
        residual[hold] = np.asarray(response[hold]) - pred_np
    return residual, fitted


def fast_split_residuals(predictor, response, prefix_folds, alpha, eta=FAST_KRR_ETA, num_folds=N_FOLDS):
    predictor = jnp.asarray(predictor, dtype=jnp.float64)
    response = jnp.asarray(response, dtype=jnp.float64)
    alpha = jnp.asarray(alpha, dtype=jnp.float64)
    n_fit = prefix_folds * len(response) // num_folds
    fit = np.arange(n_fit)
    test = np.arange(n_fit, len(response))
    pred = fixed_gaussian_krr_predict(predictor[fit], response[fit], predictor[test], alpha, eta)
    residual = np.asarray(response[test]) - np.asarray(pred)
    return np.asarray(predictor[test]), residual


def add_fast_record(records, rep, rho, direction, method, predictor, residual, seed):
    out = fast_permutation_hsic_pvalue(
        jnp.asarray(predictor, dtype=jnp.float64),
        jnp.asarray(residual, dtype=jnp.float64),
        FAST_B,
        seed,
    )
    records.append({
        "rep": rep,
        "rho": rho,
        "direction": direction,
        "method": method,
        "p_value": _float(out["p_value"]),
        "sqrt_hsic_u": _float(out["sqrt_hsic_u"]),
    })


def run_fast_repetitions(n_reps=FAST_N_REPS, n=FAST_N, seed_base=SEED + 50_000):
    records = []
    tic = time.time()
    for rep in range(n_reps):
        if rep and rep % 50 == 0:
            print(f"finished {rep}/{n_reps} fast reps in {time.time() - tic:.1f}s")
        for rho_idx, rho in enumerate(RHO_VALUES):
            data = simulate_anm(n, rho, seed=seed_base + 1_000 * rep + 100 * rho_idx)
            for dir_idx, direction in enumerate(DIRECTIONS):
                predictor, response, _ = direction_arrays(data, direction)
                mean_alpha = FAST_MEAN_ALPHA[direction]

                residual, _ = fast_full_residuals(predictor, response, mean_alpha)
                add_fast_record(
                    records, rep, rho, direction, "fast full naive",
                    predictor, residual,
                    seed_base + 1_000_000 + 1_000 * rep + 100 * rho_idx + dir_idx,
                )

                for prefix in SPLIT_PREFIXES:
                    p_test, r_test = fast_split_residuals(predictor, response, prefix, mean_alpha)
                    add_fast_record(
                        records, rep, rho, direction, f"fast {SPLIT_LABEL[prefix]} naive",
                        p_test, r_test,
                        seed_base + 2_000_000 + 1_000 * rep + 100 * rho_idx + 10 * dir_idx + prefix,
                    )
    print(f"finished {n_reps} fast reps in {time.time() - tic:.1f}s")
    return records


def run_core_repetitions(n_reps=CORE_N_REPS, n=CORE_N, config=CORE_CONFIG, seed_base=SEED + 70_000):
    records = []
    tic = time.time()
    for rep in range(n_reps):
        print(f"core rep {rep + 1}/{n_reps}")
        for rho_idx, rho in enumerate(RHO_VALUES):
            data = simulate_anm(n, rho, seed=seed_base + 1_000 * rep + 100 * rho_idx)
            for dir_idx, direction in enumerate(DIRECTIONS):
                predictor, response, signal = direction_arrays(data, direction)
                full = full_crossfit_core_test(
                    predictor, response, signal, config,
                    seed=seed_base + 1_000_000 + 1_000 * rep + 100 * rho_idx + dir_idx,
                )
                for key in ("naive", "debiased"):
                    records.append({
                        "rep": rep,
                        "rho": rho,
                        "direction": direction,
                        "method": f"core full {key}",
                        **full[key],
                    })
                for row in split_core_tests(predictor, response, signal, config):
                    records.append({"rep": rep, "rho": rho, "direction": direction, "method": f"core {row['method']}", **row})
    print(f"finished {n_reps} core reps in {time.time() - tic:.1f}s")
    return records


repetition_records = []
if RUN_FAST_REPETITIONS:
    repetition_records.extend(run_fast_repetitions())
if RUN_CORE_REPETITIONS:
    repetition_records.extend(run_core_repetitions())


In [ ]:
method_order = [
    "fast full naive",
    *[f"fast {SPLIT_LABEL[prefix]} naive" for prefix in SPLIT_PREFIXES],
    "core full naive",
    "core full debiased",
    *[f"core {SPLIT_LABEL[prefix]} naive" for prefix in SPLIT_PREFIXES],
    *[f"core {SPLIT_LABEL[prefix]} debiased" for prefix in SPLIT_PREFIXES],
]
bootstrap_method_order = ["core full naive", "core full debiased"]
method_colors = {
    "fast full naive": "#f97316",
    "core full naive": "#fb923c",
    "core full debiased": "#2563eb",
}
for i, prefix in enumerate(SPLIT_PREFIXES):
    method_colors[f"fast {SPLIT_LABEL[prefix]} naive"] = ["#737373", "#16a34a", "#a21caf", "#0f766e"][i % 4]
    method_colors[f"core {SPLIT_LABEL[prefix]} naive"] = ["#a3a3a3", "#22c55e", "#c026d3", "#14b8a6"][i % 4]
    method_colors[f"core {SPLIT_LABEL[prefix]} debiased"] = ["#93c5fd", "#60a5fa", "#3b82f6", "#1d4ed8"][i % 4]


def values_for(records, rho, direction, method, key="p_value"):
    return np.array([
        r[key] for r in records
        if r["rho"] == rho and r["direction"] == direction and r["method"] == method and key in r
    ], dtype=float)


def wilson_interval(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan, np.nan
    phat = k / n
    denom = 1.0 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = z * np.sqrt((phat * (1 - phat) + z**2 / (4 * n)) / n) / denom
    return phat, max(0.0, center - half), min(1.0, center + half)


if repetition_records:
    fig, axes = plt.subplots(len(RHO_VALUES), 2, figsize=(13, 7.2), sharey=True, constrained_layout=True)
    for row_idx, rho in enumerate(RHO_VALUES):
        for col_idx, direction in enumerate(DIRECTIONS):
            ax = axes[row_idx, col_idx]
            present_methods = [m for m in method_order if len(values_for(repetition_records, rho, direction, m, "p_value"))]
            data = [values_for(repetition_records, rho, direction, m, "p_value") for m in present_methods]
            positions = np.arange(len(present_methods))
            parts = ax.violinplot(data, positions=positions, showmeans=False, showmedians=True, widths=0.82)
            for body, method in zip(parts["bodies"], present_methods):
                body.set_facecolor(method_colors[method])
                body.set_edgecolor("#111827")
                body.set_alpha(0.45)
            for key in ["cmins", "cmaxes", "cbars", "cmedians"]:
                parts[key].set_color("#111827")
                parts[key].set_linewidth(1.0)
            ax.axhline(0.05, color="#b91c1c", linestyle="--", linewidth=1.4)
            ax.set_ylim(-0.01, 1.01)
            ax.set_xticks(positions)
            ax.set_xticklabels(present_methods, rotation=35, ha="right", fontsize=8)
            ax.set_title(f"permutation p-values: rho={rho:.2f}, {direction}")
            ax.set_ylabel("p-value")
    fig.suptitle("Permutation p-value distributions over repeated datasets", y=1.03)

    if any(len(values_for(repetition_records, rho, direction, m, "bootstrap_p_value")) for rho in RHO_VALUES for direction in DIRECTIONS for m in bootstrap_method_order):
        fig, axes = plt.subplots(len(RHO_VALUES), 2, figsize=(10.5, 6.8), sharey=True, constrained_layout=True)
        for row_idx, rho in enumerate(RHO_VALUES):
            for col_idx, direction in enumerate(DIRECTIONS):
                ax = axes[row_idx, col_idx]
                present_methods = [m for m in bootstrap_method_order if len(values_for(repetition_records, rho, direction, m, "bootstrap_p_value"))]
                data = [values_for(repetition_records, rho, direction, m, "bootstrap_p_value") for m in present_methods]
                positions = np.arange(len(present_methods))
                parts = ax.violinplot(data, positions=positions, showmeans=False, showmedians=True, widths=0.82)
                for body, method in zip(parts["bodies"], present_methods):
                    body.set_facecolor(method_colors[method])
                    body.set_edgecolor("#111827")
                    body.set_alpha(0.50)
                for key in ["cmins", "cmaxes", "cbars", "cmedians"]:
                    parts[key].set_color("#111827")
                    parts[key].set_linewidth(1.0)
                ax.axhline(0.05, color="#b91c1c", linestyle="--", linewidth=1.3)
                ax.set_ylim(-0.01, 1.01)
                ax.set_xticks(positions)
                ax.set_xticklabels(present_methods, rotation=25, ha="right", fontsize=8)
                ax.set_title(f"bootstrap p-values: rho={rho:.2f}, {direction}")
                ax.set_ylabel("RTI bootstrap p-value")
        fig.suptitle("Reverse-triangle bootstrap p-values", y=1.03)


## Main P-Value Comparison

This is the direct comparison of interest: debiased reverse-triangle bootstrap p-values versus the full merged cross-fitted HSIC permutation p-values and the ordinary fit/test split permutation p-values. All entries are plotted on the same p-value scale; the red line is 0.05.


In [ ]:
split_palette = ["#737373", "#16a34a", "#a21caf", "#0f766e", "#0891b2", "#be123c"]


def available_methods(records):
    return sorted({r["method"] for r in records})


def split_naive_methods(records):
    methods = []
    for method in available_methods(records):
        method_l = method.lower()
        if "split" in method_l and "naive" in method_l and any("p_value" in r for r in records if r["method"] == method):
            methods.append(method)
    return methods


def split_label_from_method(method):
    label = method
    for prefix in ("core ", "fast "):
        if label.startswith(prefix):
            label = label[len(prefix):]
    if label.startswith("split ") and "%" in label:
        train_pct = label.split("split ", 1)[1].split("%", 1)[0]
        return f"split plug-in, {train_pct}% train"
    return label.replace(" naive", "")


def comparison_specs_for(records):
    specs = [
        ("debiased RTI bootstrap", "core full debiased", "bootstrap_p_value", "#2563eb"),
        ("full crossfit permutation", "core full naive", "p_value", "#f97316"),
    ]
    for i, method in enumerate(split_naive_methods(records)):
        specs.append((split_label_from_method(method), method, "p_value", split_palette[i % len(split_palette)]))
    return specs


def comparison_series(records, rho, direction):
    out = []
    for label, method, key, color in comparison_specs_for(records):
        vals = values_for(records, rho, direction, method, key)
        if len(vals):
            out.append((label, vals, color))
    return out


if repetition_records:
    found_split_methods = split_naive_methods(repetition_records)
    print("Split permutation methods found:", found_split_methods if found_split_methods else "none")

    fig, axes = plt.subplots(len(RHO_VALUES), 2, figsize=(13.5, 7.4), sharey=True, constrained_layout=True)
    jitter_rng = np.random.default_rng(20260502)
    for row_idx, rho in enumerate(RHO_VALUES):
        for col_idx, direction in enumerate(DIRECTIONS):
            ax = axes[row_idx, col_idx]
            series = comparison_series(repetition_records, rho, direction)
            if not series:
                ax.text(0.5, 0.5, "no repeated records", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue
            labels = [item[0] for item in series]
            data = [item[1] for item in series]
            colors = [item[2] for item in series]
            positions = np.arange(len(series))
            parts = ax.violinplot(data, positions=positions, showmeans=False, showmedians=True, widths=0.82)
            for body, color in zip(parts["bodies"], colors):
                body.set_facecolor(color)
                body.set_edgecolor("#111827")
                body.set_alpha(0.42)
            for key in ["cmins", "cmaxes", "cbars", "cmedians"]:
                parts[key].set_color("#111827")
                parts[key].set_linewidth(1.0)
            for pos, vals, color in zip(positions, data, colors):
                jitter = jitter_rng.normal(0.0, 0.035, size=len(vals))
                ax.scatter(
                    np.full(len(vals), pos) + jitter,
                    vals,
                    s=16,
                    color=color,
                    edgecolor="#111827",
                    linewidth=0.25,
                    alpha=0.72,
                    zorder=3,
                )
            ax.axhline(0.05, color="#b91c1c", linestyle="--", linewidth=1.35)
            ax.set_ylim(-0.01, 1.01)
            ax.set_xticks(positions)
            ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
            ax.set_title(f"rho={rho:.2f}, {direction}")
            ax.set_ylabel("p-value")
    fig.suptitle("Debiased RTI bootstrap vs full/split permutation p-values", y=1.03)


In [ ]:
if repetition_records:
    fig, axes = plt.subplots(len(RHO_VALUES), 2, figsize=(13, 7.2), sharey=True, constrained_layout=True)
    for row_idx, rho in enumerate(RHO_VALUES):
        for col_idx, direction in enumerate(DIRECTIONS):
            ax = axes[row_idx, col_idx]
            present_methods = [m for m in method_order if len(values_for(repetition_records, rho, direction, m, "p_value"))]
            x = np.arange(len(present_methods))
            rates = []
            lower = []
            upper = []
            for method in present_methods:
                vals = values_for(repetition_records, rho, direction, method, "p_value")
                k = int(np.sum(vals <= 0.05))
                phat, lo, hi = wilson_interval(k, len(vals))
                rates.append(phat)
                lower.append(lo)
                upper.append(hi)
            yerr = np.vstack([np.array(rates) - np.array(lower), np.array(upper) - np.array(rates)])
            ax.bar(x, rates, color=[method_colors[m] for m in present_methods], edgecolor="#111827", linewidth=0.6)
            ax.errorbar(x, rates, yerr=yerr, fmt="none", ecolor="#111827", elinewidth=1.4, capsize=4)
            ax.axhline(0.05, color="#b91c1c", linestyle="--", linewidth=1.2)
            ax.set_ylim(0, 1.02)
            ax.set_xticks(x)
            ax.set_xticklabels(present_methods, rotation=35, ha="right", fontsize=8)
            ax.set_title(f"permutation rejection: rho={rho:.2f}, {direction}")
            ax.set_ylabel("Pr(p <= 0.05), Wilson 95% CI")
    fig.suptitle("Permutation-test rejection probabilities", y=1.03)

    if any(len(values_for(repetition_records, rho, direction, m, "bootstrap_p_value")) for rho in RHO_VALUES for direction in DIRECTIONS for m in bootstrap_method_order):
        fig, axes = plt.subplots(len(RHO_VALUES), 2, figsize=(10.5, 6.8), sharey=True, constrained_layout=True)
        for row_idx, rho in enumerate(RHO_VALUES):
            for col_idx, direction in enumerate(DIRECTIONS):
                ax = axes[row_idx, col_idx]
                present_methods = [m for m in bootstrap_method_order if len(values_for(repetition_records, rho, direction, m, "bootstrap_p_value"))]
                x = np.arange(len(present_methods))
                rates = []
                lower = []
                upper = []
                for method in present_methods:
                    vals = values_for(repetition_records, rho, direction, method, "bootstrap_p_value")
                    k = int(np.sum(vals <= 0.05))
                    phat, lo, hi = wilson_interval(k, len(vals))
                    rates.append(phat)
                    lower.append(lo)
                    upper.append(hi)
                yerr = np.vstack([np.array(rates) - np.array(lower), np.array(upper) - np.array(rates)])
                ax.bar(x, rates, color=[method_colors[m] for m in present_methods], edgecolor="#111827", linewidth=0.6)
                ax.errorbar(x, rates, yerr=yerr, fmt="none", ecolor="#111827", elinewidth=1.4, capsize=4)
                ax.axhline(0.05, color="#b91c1c", linestyle="--", linewidth=1.2)
                ax.set_ylim(0, 1.02)
                ax.set_xticks(x)
                ax.set_xticklabels(present_methods, rotation=25, ha="right", fontsize=8)
                ax.set_title(f"bootstrap rejection: rho={rho:.2f}, {direction}")
                ax.set_ylabel("Pr(bootstrap p <= 0.05), Wilson 95% CI")
        fig.suptitle("Reverse-triangle bootstrap rejection probabilities", y=1.03)


## Save Current Run

Run this cell after a big experiment to save the current in-memory results. It does not recompute anything; it just serializes `repetition_records`, `one_draw_rows`, `diagnostics`, `datasets`, and the active configuration values.


In [ ]:
from datetime import datetime
from pathlib import Path
import csv
import gzip
import json
import pickle


def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.floating, np.integer, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if hasattr(value, "shape") and hasattr(value, "dtype"):
        arr = np.asarray(value)
        return arr.item() if arr.ndim == 0 else arr.tolist()
    return value


def write_records_csv(path, records):
    if not records:
        return
    keys = sorted({key for record in records for key in record})
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=keys)
        writer.writeheader()
        for record in records:
            writer.writerow({key: json_safe(record.get(key, "")) for key in keys})


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_dir = Path("results/toy_additive_noise_gof_1d")
out_dir.mkdir(parents=True, exist_ok=True)
stem = out_dir / f"toy_additive_noise_gof_1d_{timestamp}"

payload = {
    "saved_at": timestamp,
    "settings": {
        "N_FOLDS": N_FOLDS,
        "RHO_VALUES": RHO_VALUES,
        "DIRECTIONS": DIRECTIONS,
        "SEED": SEED,
        "DEMO_N": DEMO_N,
        "DEMO_B": DEMO_B,
        "RUN_CORE_REPETITIONS": RUN_CORE_REPETITIONS,
        "CORE_N_REPS": CORE_N_REPS,
        "CORE_N": CORE_N,
        "CORE_B": CORE_B,
        "RUN_FAST_REPETITIONS": RUN_FAST_REPETITIONS,
        "FAST_N_REPS": FAST_N_REPS,
        "FAST_N": FAST_N,
        "FAST_B": FAST_B,
        "FAST_KRR_ETA": FAST_KRR_ETA,
        "FAST_MEAN_ALPHA": FAST_MEAN_ALPHA,
    },
    "demo_config": getattr(DEMO_CONFIG, "__dict__", str(DEMO_CONFIG)),
    "core_config": getattr(CORE_CONFIG, "__dict__", str(CORE_CONFIG)),
    "one_draw_rows": one_draw_rows if "one_draw_rows" in globals() else [],
    "repetition_records": repetition_records if "repetition_records" in globals() else [],
    "diagnostics": diagnostics if "diagnostics" in globals() else {},
    "datasets": datasets if "datasets" in globals() else {},
}

with gzip.open(f"{stem}.pkl.gz", "wb") as handle:
    pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(f"{stem}.json", "w", encoding="utf-8") as handle:
    json.dump(json_safe(payload), handle, indent=2)

write_records_csv(Path(f"{stem}_repetition_records.csv"), payload["repetition_records"])
write_records_csv(Path(f"{stem}_one_draw_rows.csv"), payload["one_draw_rows"])

print(f"Saved run to {stem}.pkl.gz")
print(f"Saved JSON summary to {stem}.json")
if payload["repetition_records"]:
    print(f"Saved repetition CSV to {stem}_repetition_records.csv")
if payload["one_draw_rows"]:
    print(f"Saved one-draw CSV to {stem}_one_draw_rows.csv")


## Load And Visualize Saved Runs

Use these cells to inspect a saved run without rerunning the experiment. By default, `load_saved_run()` loads the most recent timestamped pickle from `results/toy_additive_noise_gof_1d/`.


In [ ]:
from pathlib import Path
import gzip
import pickle


def list_saved_runs(results_dir="results/toy_additive_noise_gof_1d"):
    results_dir = Path(results_dir)
    runs = sorted(results_dir.glob("toy_additive_noise_gof_1d_*.pkl.gz"))
    if not runs:
        print(f"No saved runs found in {results_dir.resolve()}")
        return []
    for idx, path in enumerate(runs):
        print(f"{idx:>2}: {path.name}")
    return runs


def load_saved_run(path=None, results_dir="results/toy_additive_noise_gof_1d"):
    runs = list_saved_runs(results_dir)
    if path is None:
        if not runs:
            raise FileNotFoundError(f"No saved runs found in {Path(results_dir).resolve()}")
        path = runs[-1]
    else:
        path = Path(path)
    with gzip.open(path, "rb") as handle:
        payload = pickle.load(handle)
    print(f"Loaded {path}")
    print(f"saved_at: {payload.get('saved_at', 'unknown')}")
    print(f"repetition records: {len(payload.get('repetition_records', []))}")
    print(f"one-draw rows: {len(payload.get('one_draw_rows', []))}")
    return payload


# Set an explicit existing payload for analysis only; None uses the run just saved above.
SAVED_RUN_PATH = None
if SAVED_RUN_PATH is None:
    if "stem" not in globals():
        raise ValueError("Set SAVED_RUN_PATH to the specific .pkl.gz run to analyse.")
    SAVED_RUN_PATH = f"{stem}.pkl.gz"
loaded_run = load_saved_run(SAVED_RUN_PATH)
loaded_records = loaded_run.get("repetition_records", [])
loaded_one_draw_rows = loaded_run.get("one_draw_rows", [])

# These also permit replaying the earlier Forward And Reverse Fits figure.
datasets = loaded_run["datasets"]
diagnostics = loaded_run["diagnostics"]


In [ ]:
def saved_values_for(records, rho, direction, method, key="p_value"):
    return np.array([
        r[key] for r in records
        if r.get("rho") == rho and r.get("direction") == direction and r.get("method") == method and key in r
    ], dtype=float)


def saved_wilson_interval(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan, np.nan
    phat = k / n
    denom = 1.0 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = z * np.sqrt((phat * (1 - phat) + z**2 / (4 * n)) / n) / denom
    return phat, max(0.0, center - half), min(1.0, center + half)


def saved_rhos(records):
    return sorted({float(r["rho"]) for r in records if "rho" in r})


def saved_directions(records):
    preferred = ["Y on X", "X on Y"]
    found = {r["direction"] for r in records if "direction" in r}
    ordered = [direction for direction in preferred if direction in found]
    ordered.extend(sorted(found - set(ordered)))
    return ordered


def saved_available_methods(records):
    return sorted({r["method"] for r in records if "method" in r})


def saved_split_naive_methods(records):
    methods = []
    for method in saved_available_methods(records):
        method_l = method.lower()
        if "split" in method_l and "naive" in method_l and any("p_value" in r for r in records if r.get("method") == method):
            methods.append(method)
    return methods


def saved_split_label_from_method(method):
    label = method
    for prefix in ("core ", "fast "):
        if label.startswith(prefix):
            label = label[len(prefix):]
    return label.replace(" naive", " fit/test perm")


def saved_comparison_specs(records):
    split_palette = ["#737373", "#16a34a", "#a21caf", "#0f766e", "#0891b2", "#be123c"]
    specs = [
        ("debiased RTI bootstrap", "core full debiased", "bootstrap_p_value", "#2563eb"),
        ("full crossfit permutation", "core full naive", "p_value", "#f97316"),
    ]
    for i, method in enumerate(saved_split_naive_methods(records)):
        specs.append((saved_split_label_from_method(method), method, "p_value", split_palette[i % len(split_palette)]))
    return specs


def saved_comparison_series(records, rho, direction):
    out = []
    for label, method, key, color in saved_comparison_specs(records):
        vals = saved_values_for(records, rho, direction, method, key)
        if len(vals):
            out.append((label, vals, color))
    return out


import pandas as pd


def saved_rejection_curve(vals, alpha_grid, z=1.96):
    vals = np.sort(np.asarray(vals, dtype=float))
    vals = vals[np.isfinite(vals)]
    n = len(vals)
    if n == 0:
        nan = np.full_like(alpha_grid, np.nan, dtype=float)
        return nan, nan, nan
    counts = np.searchsorted(vals, alpha_grid, side="right")
    phat = counts / n
    denom = 1.0 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = z * np.sqrt((phat * (1 - phat) + z**2 / (4 * n)) / n) / denom
    return phat, np.maximum(0.0, center - half), np.minimum(1.0, center + half)


def saved_plot_label(label):
    replacements = {
        "debiased RTI bootstrap": "debiased bootstrap",
        "full crossfit permutation": "cross-fit permutation",
        "split plug-in, 20% train": "20% split",
        "split plug-in, 40% train": "40% split",
        "split plug-in, 60% train": "60% split",
        "split plug-in, 80% train": "80% split",
        "split 20% fit/test perm": "20% split",
        "split 40% fit/test perm": "40% split",
        "split 60% fit/test perm": "60% split",
        "split 80% fit/test perm": "80% split",
    }
    return replacements.get(label, label)


def saved_method_code(label):
    short_label = saved_plot_label(label)
    codes = {
        "debiased bootstrap": "DB",
        "cross-fit permutation": "CF",
        "20% split": "S20",
        "40% split": "S40",
        "60% split": "S60",
        "80% split": "S80",
    }
    return codes.get(short_label, "")


def saved_method_axis_label(label):
    short_label = saved_plot_label(label)
    axis_labels = {
        "debiased bootstrap": "debiased",
        "cross-fit permutation": "cross-fit",
        "20% split": "split 20%",
        "40% split": "split 40%",
        "60% split": "split 60%",
        "80% split": "split 80%",
    }
    return axis_labels.get(short_label, short_label)


def saved_panel_title(rho, direction):
    if "panel_title" in globals():
        return panel_title(rho, direction)
    direction_label = r"$Y$ on $X$" if direction == "Y on X" else r"$X$ on $Y$"
    status = "null holds" if direction == "Y on X" and np.isclose(rho, 0.0) else "null fails"
    return f"{direction_label}: {status}"


def plot_saved_pvalue_violins(records, show_title=True):
    rhos = saved_rhos(records)
    directions = saved_directions(records)
    if not records or not rhos or not directions:
        print("No repeated records available to plot.")
        return
    fig, axes = plt.subplots(len(rhos), len(directions), figsize=(6.2 * len(directions), 2.55 * len(rhos)), sharex=True, sharey=True, constrained_layout=False)
    fig.subplots_adjust(left=0.065, right=0.995, bottom=0.22, top=0.88 if show_title else 0.94, wspace=0.035, hspace=0.34)
    axes = np.asarray(axes).reshape(len(rhos), len(directions))
    for row_idx, rho in enumerate(rhos):
        for col_idx, direction in enumerate(directions):
            ax = axes[row_idx, col_idx]
            series = saved_comparison_series(records, rho, direction)
            plugin = saved_values_for(records, rho, direction, "core full naive", "bootstrap_p_value")
            if len(plugin):
                series.insert(1, ("plug-in bootstrap", plugin, "#4f46e5"))
            if not series:
                ax.text(0.5, 0.5, "no records", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue
            labels = []
            for label, _, _ in series:
                if label == "debiased RTI bootstrap":
                    label = "debiased bootstrap"
                elif label == "full crossfit permutation":
                    label = "cross-fit plug-in permutation"
                elif label.startswith("split "):
                    percentage = label.split()[1]
                    kind = "permutation" if percentage == "40%" else "plug-in"
                    label = f"split {kind}, {percentage} train"
                labels.append(label)
            data = [item[1] for item in series]
            colors = [item[2] for item in series]
            positions = np.arange(len(series))
            parts = ax.violinplot(data, positions=positions, showmeans=False, showmedians=True, widths=0.78)
            for body, color in zip(parts["bodies"], colors):
                body.set_facecolor(color)
                body.set_edgecolor("#111827")
                body.set_alpha(0.46)
            for key in ["cmins", "cmaxes", "cbars", "cmedians"]:
                parts[key].set_color("#111827")
                parts[key].set_linewidth(0.9)
            ax.axhline(0.05, color="#b91c1c", linestyle="--", linewidth=1.05, alpha=0.75)
            ax.set_ylim(-0.01, 1.01)
            ax.set_xticks(positions)
            if row_idx == len(rhos) - 1:
                ax.set_xticklabels(labels, rotation=25, ha="right", fontsize=8.5)
            else:
                ax.set_xticklabels([])
            ax.set_title(saved_panel_title(rho, direction), fontsize=11, fontweight="semibold")
            if col_idx == 0:
                ax.set_ylabel(r"$p$-value")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
    if show_title:
        fig.suptitle(r"$p$-value distributions", y=1.02)

    return fig


def plot_saved_rejection_snapshot(records, alpha=0.05, show_title=True):
    rhos = saved_rhos(records)
    directions = saved_directions(records)
    if not records or not rhos or not directions:
        print("No repeated records available to plot.")
        return
    print("Split permutation methods found:", saved_split_naive_methods(records) or "none")

    fig, axes = plt.subplots(len(rhos), len(directions), figsize=(4.45 * len(directions), 2.25 * len(rhos)), sharex=True, constrained_layout=False)
    fig.subplots_adjust(left=0.16, right=0.995, bottom=0.10, top=0.90, wspace=0.08, hspace=0.34)
    axes = np.asarray(axes).reshape(len(rhos), len(directions))
    for row_idx, rho in enumerate(rhos):
        for col_idx, direction in enumerate(directions):
            ax = axes[row_idx, col_idx]
            series = saved_comparison_series(records, rho, direction)
            if not series:
                ax.text(0.5, 0.5, "no records", ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()
                continue
            labels = [saved_plot_label(label) for label, _, _ in series]
            y = np.arange(len(series))[::-1]
            for pos, (label, vals, color) in zip(y, series):
                vals = np.asarray(vals, dtype=float)
                vals = vals[np.isfinite(vals)]
                rate, lo, hi = [arr[0] for arr in saved_rejection_curve(vals, np.array([alpha], dtype=float))]
                xerr = np.array([[max(0.0, rate - lo)], [max(0.0, hi - rate)]])
                ax.errorbar(rate, pos, xerr=xerr, fmt="o", markersize=6.5, color=color, ecolor=color, elinewidth=1.45, capsize=3.5, markeredgecolor="#111827", markeredgewidth=0.55, zorder=3)
            ax.axvline(alpha, color="#b91c1c", linestyle="--", linewidth=1.25, alpha=0.8)
            ax.set_xlim(-0.02, 1.04)
            ax.set_yticks(y)
            if col_idx == 0:
                ax.set_yticklabels(labels, fontsize=10.5)
            else:
                ax.set_yticklabels([])
            ax.set_xticks(np.linspace(0, 1, 6))
            ax.set_xticklabels([f"{100 * tick:.0f}%" for tick in np.linspace(0, 1, 6)], fontsize=9.5)
            ax.set_title(saved_panel_title(rho, direction), fontsize=11)
            ax.grid(axis="x", color="#e5e7eb", linewidth=0.8)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
    if show_title:
        fig.suptitle(rf"Rejection probability at $\alpha={alpha:g}$", y=1.02)
    return fig


def saved_dgp_panel_title(rho):
    label = "homoskedastic DGP" if np.isclose(rho, 0.0) else "heteroskedastic DGP"
    return rf"{label} ($\rho={rho:g}$)"


def saved_direction_label(direction):
    if direction == "Y on X":
        return r"$X\to Y$"
    if direction == "X on Y":
        return r"$Y\to X$"
    return direction


SAVED_NULL_STATUS_COLORS = {"null holds": "#2563eb", "null fails": "#dc2626"}


def saved_null_status_for(rho, direction):
    if "null_status_label" in globals():
        return null_status_label(rho, direction)
    return "null holds" if direction == "Y on X" and np.isclose(rho, 0.0) else "null fails"


def plot_saved_rejection_snapshot_compact(records, alpha=0.05, show_title=True):
    rhos = saved_rhos(records)
    directions = saved_directions(records)
    if not records or not rhos or not directions:
        print("No repeated records available to plot.")
        return
    print("Split permutation methods found:", saved_split_naive_methods(records) or "none")

    panels = [(rho, direction) for rho in rhos for direction in directions]
    specs = [
        (label, method, key, color)
        for label, method, key, color in saved_comparison_specs(records)
        if any(len(saved_values_for(records, rho, direction, method, key)) for rho, direction in panels)
    ]
    if not specs:
        print("No repeated records available to plot.")
        return

    labels = [saved_method_axis_label(label) for label, _, _, _ in specs]
    method_colors = [color for _, _, _, color in specs]
    y = np.arange(len(specs))[::-1]
    panel_count = len(panels)
    fig_width = max(9.2, 2.55 * panel_count + 1.45)
    fig_height = max(2.85, 0.34 * len(specs) + 1.45)
    fig, axes = plt.subplots(
        1,
        panel_count,
        figsize=(fig_width, fig_height),
        sharex=True,
        sharey=True,
        constrained_layout=False,
    )
    axes = np.asarray(axes).reshape(panel_count)
    fig.subplots_adjust(
        left=0.105,
        right=0.995,
        bottom=0.17,
        top=0.84 if show_title else 0.90,
        wspace=0.07,
    )

    for panel_idx, (ax, (rho, direction)) in enumerate(zip(axes, panels)):
        for band_idx, pos in enumerate(y):
            if band_idx % 2 == 0:
                ax.axhspan(pos - 0.5, pos + 0.5, color="#f8fafc", alpha=0.78, zorder=-2)
        for separator in np.arange(0.5, len(specs) - 0.5, 1.0):
            ax.axhline(separator, color="#cbd5e1", linewidth=0.55, alpha=0.74, zorder=0)

        for pos, (label, method, key, color) in zip(y, specs):
            vals = saved_values_for(records, rho, direction, method, key)
            vals = vals[np.isfinite(vals)]
            if not len(vals):
                continue
            rate, lo, hi = [arr[0] for arr in saved_rejection_curve(vals, np.array([alpha], dtype=float))]
            xerr = np.array([[max(0.0, rate - lo)], [max(0.0, hi - rate)]])
            ax.errorbar(
                rate,
                pos,
                xerr=xerr,
                fmt="o",
                markersize=6.2,
                color=color,
                ecolor=color,
                elinewidth=1.25,
                capsize=3.0,
                markeredgecolor="#111827",
                markeredgewidth=0.65,
                zorder=3,
            )

        ax.axvline(alpha, color="#b91c1c", linestyle="--", linewidth=1.1, alpha=0.82)
        ax.set_xlim(-0.02, 1.04)
        ax.set_ylim(-0.55, len(specs) - 0.45)
        ax.set_yticks(y)
        if panel_idx == 0:
            ax.set_yticklabels(labels, fontsize=8.8)
            for tick, color in zip(ax.get_yticklabels(), method_colors):
                tick.set_color(color)
                tick.set_fontweight("semibold")
        else:
            ax.tick_params(axis="y", which="both", left=False, labelleft=False)

        status_label = saved_null_status_for(rho, direction)
        status_short = "null" if status_label == "null holds" else "alt"
        ax.set_title(
            rf"$\rho={rho:g}$, {saved_direction_label(direction)} ({status_label})",
            loc="left",
            pad=3,
            fontsize=12,
            fontweight="semibold",
        )
        ax.set_xticks(np.linspace(0, 1, 5))
        ax.set_xticklabels([f"{100 * tick:.0f}%" for tick in np.linspace(0, 1, 5)], fontsize=8.8)
        ax.grid(axis="x", color="#e5e7eb", linewidth=0.72)
        ax.grid(False, axis="y")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    alpha_pct = f"{100 * alpha:g}%"
    fig.supxlabel("rejection probability", x=0.58, y=0.045, fontsize=12)
    if show_title:
        fig.suptitle(f"Rejection probability at {alpha_pct}", y=0.995, fontsize=11.0, fontweight="semibold")
    return fig


def saved_rejection_table(records, alpha_marks=(0.01, 0.025, 0.05, 0.10, 0.20)):
    rows = []
    alpha_marks = np.array(alpha_marks, dtype=float)
    for rho in saved_rhos(records):
        for direction in saved_directions(records):
            for label, vals, _ in saved_comparison_series(records, rho, direction):
                vals = np.asarray(vals, dtype=float)
                vals = vals[np.isfinite(vals)]
                rates, lower, upper = saved_rejection_curve(vals, alpha_marks)
                for alpha, rate, lo, hi in zip(alpha_marks, rates, lower, upper):
                    rows.append({
                        "scenario": saved_panel_title(rho, direction).replace("$", ""),
                        "rho": rho,
                        "direction": direction,
                        "method": saved_plot_label(label),
                        "alpha": alpha,
                        "n": len(vals),
                        "rejection_probability": rate,
                        "wilson_lower": lo,
                        "wilson_upper": hi,
                        "estimate_95_ci": f"{100 * rate:.0f} [{100 * lo:.0f}, {100 * hi:.0f}]",
                    })
    return pd.DataFrame(rows)


def display_saved_rejection_table(records, alpha_marks=(0.01, 0.025, 0.05, 0.10, 0.20)):
    table = saved_rejection_table(records, alpha_marks=alpha_marks)
    if table.empty:
        print("No repeated records available for table.")
        return table
    formatted = table.pivot_table(
        index=["rho", "scenario", "method", "n"],
        columns="alpha",
        values="estimate_95_ci",
        aggfunc="first",
    )
    formatted.columns = [f"alpha={alpha:g}" for alpha in formatted.columns]
    display(formatted)
    return table


saved_violin_fig = plot_saved_pvalue_violins(loaded_records, show_title=False)
with plt.style.context("default"), plt.rc_context({"font.family": "DejaVu Sans"}):
    saved_compact_rejection_fig = plot_saved_rejection_snapshot_compact(loaded_records, alpha=0.05, show_title=True)
loaded_rejection_table = display_saved_rejection_table(loaded_records)

figure_dir = CAUSAL_FIGURE_DIR
figure_dir.mkdir(parents=True, exist_ok=True)
saved_violin_fig.savefig(figure_dir / "causal_arrow_violin_no_title.png", dpi=100, bbox_inches="tight")
for extension in ("pdf", "png"):
    saved_compact_rejection_fig.savefig(figure_dir / f"causal_arrow_rej_prob_compact.{extension}", dpi=300, bbox_inches="tight")
loaded_rejection_table.to_csv(figure_dir / "causal_arrow_rejection_summary.csv", index=False)
